# Check the extracted files' contents by the extracted skills by using API

In [3]:
import pandas as pd
import re

def clean_csv_skills(skill_string):
    """Parses messy CSV strings like "{['skill 1', 'skill 2']}" into a set of lowercase strings."""
    if pd.isna(skill_string): return set()
    cleaned_str = str(skill_string).strip("[]{} ")
    if not cleaned_str: return set()
    skills = [re.sub(r"^['\"]+|['\"]+$", "", s.strip()) for s in cleaned_str.split(',')]
    return set(s.lower() for s in skills if s)

def clean_xlsx_skills(skill_string):
    """Parses XLSX strings like "skill 1,skill 2" into a set of lowercase strings."""
    if pd.isna(skill_string): return set()
    skills = str(skill_string).split(',')
    return set(s.strip().lower() for s in skills if s.strip())

# ==========================================
# --- Configurations ---
# ==========================================
CSV_FILE = '/Users/ryanlin/Downloads/Resume_with_skills_10hr.csv'
XLSX_FILE = '/Users/ryanlin/Downloads/resumes_with_skills_cleaned.xlsx'

ID_COL = 'ID'

# Updated to match your exact column headers
ACTUAL_CSV_SKILL_HEADER = 'Extracted_Skills'  
ACTUAL_XLSX_SKILL_HEADER = 'skills_extracted' 
# ==========================================

print("Loading datasets...")
df_csv = pd.read_csv(CSV_FILE)
df_xlsx = pd.read_excel(XLSX_FILE)

# Rename the columns to standard internal names BEFORE merging to avoid KeyErrors
df_csv = df_csv.rename(columns={ACTUAL_CSV_SKILL_HEADER: 'csv_target_skills'})
df_xlsx = df_xlsx.rename(columns={ACTUAL_XLSX_SKILL_HEADER: 'xlsx_target_skills'})

# Merge the dataframes on ID
merged_df = pd.merge(df_csv, df_xlsx, on=ID_COL, how='inner')

results = []
raw_accuracies = [] 

for index, row in merged_df.iterrows():
    current_id = row[ID_COL]
    
    # We now use our guaranteed internal column names
    skills_csv = clean_csv_skills(row['csv_target_skills'])
    skills_xlsx = clean_xlsx_skills(row['xlsx_target_skills'])
    
    # Skip this ID entirely if either list of skills ends up empty
    if not skills_csv or not skills_xlsx:
        continue
        
    # Set Mathematics
    common_skills = skills_csv.intersection(skills_xlsx)
    all_unique_skills = skills_csv.union(skills_xlsx)
    
    # Accuracy per ID (Overlap divided by total unique skills)
    if len(all_unique_skills) > 0:
        accuracy = (len(common_skills) / len(all_unique_skills)) * 100
    else:
        continue # Skip if somehow there are 0 unique skills combined
        
    raw_accuracies.append(accuracy)
    
    # Mismatches
    mismatch_csv = skills_csv - skills_xlsx 
    mismatch_xlsx = skills_xlsx - skills_csv 
    
    results.append({
        'ID': current_id,
        'Accuracy': f"{accuracy:.2f}%",
        'the skills does not match from table a': ", ".join(mismatch_csv) if mismatch_csv else "All Match",
        'the skills does not match from table b': ", ".join(mismatch_xlsx) if mismatch_xlsx else "All Match"
    })

# Create the final DataFrame
report_df = pd.DataFrame(results)

# --- Display Results ---
if not report_df.empty:
    overall_average = sum(raw_accuracies) / len(raw_accuracies)
    print("\n" + "="*40)
    print(f"Total Valid IDs Processed: {len(report_df)}")
    print(f"OVERALL AVERAGE ACCURACY:  {overall_average:.2f}%")
    print("="*40 + "\n")
    
    print("DataFrame Preview (First 5 rows):")
    print(report_df.head(5).to_string(index=False)) 
else:
    print("\nNo valid IDs remained to process. Check your data.")

Loading datasets...

Total Valid IDs Processed: 2438
OVERALL AVERAGE ACCURACY:  10.02%

DataFrame Preview (First 5 rows):
      ID Accuracy                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [4]:
# Run this ONLY AFTER you have checked the output of Part 1

OUTPUT_FILE = '/Users/ryanlin/Downloads/skill_mismatches_final.xlsx'

if 'report_df' in locals() and not report_df.empty:
    print(f"Exporting {len(report_df)} rows to Excel...")
    report_df.to_excel(OUTPUT_FILE, index=False)
    print(f"Success! Data successfully exported to:\n{OUTPUT_FILE}")
else:
    print("Error: The dataframe 'report_df' is empty or does not exist. Please run Part 1 successfully first.")

Exporting 2438 rows to Excel...
Success! Data successfully exported to:
/Users/ryanlin/Downloads/skill_mismatches_final.xlsx


'skills_csv'